## MLflow trong Deep Learning

#### Nếu chwua tải về, bạn có thể cài đặt MLflow bằng pip:

In [ ]:
# pip install mlflow torch torchvision

### Step 1: Tạo một MLflow Experiment

In [2]:
import mlflow

# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_experiment("Deep Learning Experiment")

# IMPORTANT: Enable system metrics monitoring
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

/conda/miniconda3/envs/voice/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)


### Step 2: Chuẩn bị dữ liệu

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and prepare data
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_dataset = datasets.FashionMNIST("data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST("data", train=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

### Step 3: Định nghĩa mô hình và hàm tối ưu

In [4]:
import torch.nn as nn


class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)

In [ ]:
# Training parameters
params = {
    "epochs": 10,
    "learning_rate": 7.5e-4,
    "batch_size": 128,
    "optimizer": "SGD",
    "model_type": "MLP",
    "hidden_units": [512, 512],
}

# Define optimizer and loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"])

### Step 4: Huấn luyện mô hình và log kết quả với MLflow

In [9]:
with mlflow.start_run() as run:
    # Log training parameters
    mlflow.log_params(params)

    for epoch in range(params["epochs"]):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Forward pass
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Calculate metrics
            train_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Log batch metrics (every 100 batches)
            if batch_idx % 100 == 0:
                batch_loss = train_loss / (batch_idx + 1)
                batch_acc = 100.0 * correct / total
                mlflow.log_metrics(
                    {"batch_loss": batch_loss, "batch_accuracy": batch_acc},
                    step=epoch * len(train_loader) + batch_idx,
                )

        # Calculate epoch metrics
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = loss_fn(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        # Calculate and log epoch validation metrics
        val_loss = val_loss / len(test_loader)
        val_acc = 100.0 * val_correct / val_total

        # Log epoch metrics
        mlflow.log_metrics(
            {
                "train_loss": epoch_loss,
                "train_accuracy": epoch_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            },
            step=epoch,
        )
        # Log checkpoint at the end of each epoch
        mlflow.pytorch.log_model(model, name=f"checkpoint_{epoch}")

        print(
            f"Epoch {epoch + 1}/{params['epochs']}, "
            f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
        )

    # Log the final trained model
    model_info = mlflow.pytorch.log_model(model, name="final_model")

2026/03/30 13:41:22 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Epoch 1/10, Train Loss: 0.6305, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 2/10, Train Loss: 0.6305, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 3/10, Train Loss: 0.6306, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 4/10, Train Loss: 0.6304, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 5/10, Train Loss: 0.6305, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 6/10, Train Loss: 0.6304, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 7/10, Train Loss: 0.6305, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 8/10, Train Loss: 0.6303, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 9/10, Train Loss: 0.6305, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%
Epoch 10/10, Train Loss: 0.6305, Train Acc: 78.84%, Val Loss: 0.6476, Val Acc: 77.82%


2026/03/30 13:44:26 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/30 13:44:26 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


### Tiếp tục huấn luyện và log kết quả

In [ ]:
model = mlflow.pytorch.load_model(f"runs:/df99bbc168e34a0ca930b7f13e04108a/final_model").to(device) # với run_id = "df99bbc168e34a0ca930b7f13e04108a"
optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"])

In [11]:
with mlflow.start_run() as run:
    # Log training parameters
    mlflow.log_params(params)

    for epoch in range(params["epochs"]):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Forward pass
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Calculate metrics
            train_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Log batch metrics (every 100 batches)
            if batch_idx % 100 == 0:
                batch_loss = train_loss / (batch_idx + 1)
                batch_acc = 100.0 * correct / total
                mlflow.log_metrics(
                    {"batch_loss": batch_loss, "batch_accuracy": batch_acc},
                    step=epoch * len(train_loader) + batch_idx,
                )

        # Calculate epoch metrics
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = loss_fn(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        # Calculate and log epoch validation metrics
        val_loss = val_loss / len(test_loader)
        val_acc = 100.0 * val_correct / val_total

        # Log epoch metrics
        mlflow.log_metrics(
            {
                "train_loss": epoch_loss,
                "train_accuracy": epoch_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            },
            step=epoch,
        )
        # Log checkpoint at the end of each epoch
        mlflow.pytorch.log_model(model, name=f"checkpoint_{epoch}")

        print(
            f"Epoch {epoch + 1}/{params['epochs']}, "
            f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
        )

    # Log the final trained model
    model_info = mlflow.pytorch.log_model(model, name="final_model")

2026/03/30 13:50:51 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Epoch 1/10, Train Loss: 0.6202, Train Acc: 79.13%, Val Loss: 0.6282, Val Acc: 78.55%
Epoch 2/10, Train Loss: 0.6014, Train Acc: 79.65%, Val Loss: 0.6108, Val Acc: 79.02%
Epoch 3/10, Train Loss: 0.5848, Train Acc: 80.06%, Val Loss: 0.5974, Val Acc: 79.23%
Epoch 4/10, Train Loss: 0.5708, Train Acc: 80.59%, Val Loss: 0.5839, Val Acc: 79.70%
Epoch 5/10, Train Loss: 0.5583, Train Acc: 80.90%, Val Loss: 0.5732, Val Acc: 79.87%
Epoch 6/10, Train Loss: 0.5474, Train Acc: 81.25%, Val Loss: 0.5634, Val Acc: 80.35%
Epoch 7/10, Train Loss: 0.5378, Train Acc: 81.56%, Val Loss: 0.5550, Val Acc: 80.53%
Epoch 8/10, Train Loss: 0.5286, Train Acc: 81.81%, Val Loss: 0.5496, Val Acc: 80.77%
Epoch 9/10, Train Loss: 0.5209, Train Acc: 82.00%, Val Loss: 0.5401, Val Acc: 80.91%
Epoch 10/10, Train Loss: 0.5139, Train Acc: 82.27%, Val Loss: 0.5342, Val Acc: 81.04%


2026/03/30 13:53:58 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/30 13:53:58 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


### Step 5: Theo dõi và so sánh các runs trong MLflow UI

In [7]:
# !mlflow server --port 5000

### Step 6: Load mô hình đã lưu và sử dụng

In [8]:
# Load the final model
model = mlflow.pytorch.load_model("runs:/df99bbc168e34a0ca930b7f13e04108a/final_model") # .load_model("runs:/<run_id>/final_model")
# or load a checkpoint
# model = mlflow.pytorch.load_model("runs:/<run_id>/checkpoint_<epoch>")
model.to(device)
model.eval()

# Resume the previous run to log test metrics
with mlflow.start_run(run_id=run.info.run_id) as run:
    # Evaluate the model on the test set
    test_loss, test_correct, test_total = 0, 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = loss_fn(output, target)

            test_loss += loss.item()
            _, predicted = output.max(1)
            test_total += target.size(0)
            test_correct += predicted.eq(target).sum().item()

    # Calculate and log final test metrics
    test_loss = test_loss / len(test_loader)
    test_acc = 100.0 * test_correct / test_total

    mlflow.log_metrics({"test_loss": test_loss, "test_accuracy": test_acc})
    print(f"Final Test Accuracy: {test_acc:.2f}%")

2026/03/30 13:37:27 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/03/30 13:37:28 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/30 13:37:28 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Final Test Accuracy: 77.82%


### Cụ thể với các framework, mlflow đều có các module riêng để log và load mô hình, xem chi tiết tại:
- [MLflow PyTorch](https://mlflow.org/docs/latest/ml/deep-learning/pytorch/)
- [MLflow TensorFlow](https://mlflow.org/docs/latest/ml/deep-learning/tensorflow/)
- [MLflow Keras](https://mlflow.org/docs/latest/ml/deep-learning/keras/)
- [MLflow Transformer](https://mlflow.org/docs/latest/ml/deep-learning/transformers/)
- [MLflow Sentence Transformers](https://mlflow.org/docs/latest/ml/deep-learning/sentence-transformers/)    
- [MLflow spaCy](https://mlflow.org/docs/latest/ml/deep-learning/spacy/)
